# Week 2: AI for Academic Research
**Applied Generative AI**

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/)

*Developed by Prachi Patel, PhD | Open Curriculum — CC BY 4.0*

---

## Learning Objectives

By the end of this notebook, you will be able to:

- **Navigate** the landscape of AI research tools and understand their strengths/limitations
- **Design** effective research prompts for literature review and synthesis
- **Implement** a verification workflow for AI-generated citations and claims
- **Apply** cross-validation techniques using multiple AI research tools
- **Practice** responsible academic use of AI with proper disclosure

---
## Setup

In [ ]:
!pip install -q openai requests beautifulsoup4 pandas

In [ ]:
import requests
import pandas as pd
from openai import OpenAI
import getpass
import json

print("✅ Setup complete!")

---
## 1. The AI Research Tools Landscape

A growing ecosystem of AI-powered tools can accelerate academic research — but each has distinct strengths and limitations. Understanding this landscape helps you choose the right tool for each research task.

### Key Tools Overview

| Tool | What It Does Well | What It Does Poorly |
|------|-------------------|---------------------|
| **ChatGPT Deep Research** | Multi-step reasoning, synthesizes across sources, conversational follow-up | Citations can be hallucinated; paywall for advanced features |
| **Claude** | Long context (200K tokens), nuanced analysis, strong at literature synthesis | No native real-time search; relies on training data cutoff |
| **Gemini Deep Research** | Google Scholar integration, real-time web search, structured reports | Newer tool; citation quality varies |
| **Perplexity** | Real-time web search, inline citations, free tier available | Shallow on niche academic topics; mixed source quality |
| **Elicit** | Built for academic research, paper discovery, literature mapping | Focused on papers; less flexible for open-ended synthesis |
| **Semantic Scholar** | Free API, 200M+ papers, citation graph, no key required | No synthesis; raw data only — you build the workflow |
| **Connected Papers** | Visual citation graphs, "prior" and "derivative" papers | Discovery-focused; not for synthesis or Q&A |

### When to Use Which Tool

- **Broad survey of a field** → Elicit, Semantic Scholar, Connected Papers (discovery) + Claude or ChatGPT (synthesis)
- **Specific factual question** → Perplexity (real-time) or Gemini Deep Research
- **Gap identification** → Elicit + manual verification
- **Literature review draft** → ChatGPT/Claude with **strict verification** of every citation
- **Building custom pipelines** → Semantic Scholar API (free, programmable)

---
## 2. Prompt Design for Research Queries

How you structure your research prompt dramatically affects output quality. Below we demonstrate three strategies: **broad survey**, **specific question**, and **gap identification** — and show how a structured template improves results.

### Research Prompt Strategies

| Strategy | Use Case | Example |
|----------|----------|---------|
| **Broad survey** | Mapping a field, finding key papers | "Summarize the main approaches to few-shot learning in NLP from 2019–2024" |
| **Specific question** | Answering a precise research question | "What is the state-of-the-art accuracy for named entity recognition on the CoNLL-2003 benchmark?" |
| **Gap identification** | Finding research opportunities | "What limitations do current methods for long-context reasoning have, and what has not yet been tried?" |

A **structured template** improves consistency: define **scope** (topic, time range, domain), **depth** (overview vs. detailed), and **format** (bullet points, narrative, table).

In [ ]:
# Structured research query template
def build_research_prompt(topic: str, scope: str = "2019-2024", depth: str = "overview", 
                         format_instruction: str = "bullet points") -> str:
    """Build a structured research query for better LLM output."""
    return f"""You are an academic research assistant. Provide a rigorous, well-sourced response.

TOPIC: {topic}
SCOPE: Focus on literature from {scope}. Cite specific papers where possible.
DEPTH: {depth}
FORMAT: Respond in {format_instruction}. For each key claim, note if you are citing a real paper or summarizing general knowledge.

If citing a paper, use format: Author et al. (Year) - Title. Only cite papers you are confident exist.
If uncertain, say "General consensus suggests..." rather than inventing a citation."""

# Example: different prompt strategies
broad_survey = build_research_prompt("few-shot learning in NLP", scope="2019-2024", depth="overview")
specific_question = build_research_prompt("state-of-the-art NER accuracy on CoNLL-2003", scope="2020-2024", depth="detailed")
gap_id = build_research_prompt("limitations of long-context reasoning methods and unexplored approaches", depth="analytical")

print("=== Broad Survey Prompt ===\n", broad_survey[:400], "...\n")
print("=== Specific Question Prompt ===\n", specific_question[:400], "...")

In [ ]:
# OpenAI API demo: research prompting (requires API key)
# Run this cell and enter your key when prompted (use getpass for Colab security)
try:
    api_key = getpass.getpass("Enter your OpenAI API key (or press Enter to skip): ")
    if api_key:
        client = OpenAI(api_key=api_key)
        response = client.chat.completions.create(
            model="gpt-4o-mini",  # Cost-effective for demos; use gpt-4o for production research
            messages=[
                {"role": "system", "content": "You are an academic research assistant. Be concise. Only cite papers you are confident exist."},
                {"role": "user", "content": build_research_prompt("retrieval-augmented generation (RAG)", scope="2020-2024", depth="overview")}
            ],
            max_tokens=500
        )
        print("=== Sample Research Output ===\n")
        print(response.choices[0].message.content)
    else:
        print("Skipped: No API key provided. Run again with a key to see live output.")
except Exception as e:
    print(f"Error: {e}. Ensure openai is installed and you have a valid API key.")

---
## 3. The Verification Workflow

AI models can generate **convincing citations to papers that don't exist** — a well-documented problem called *citation hallucination*. The solution: **verify every citation** against a trusted source. We use the **Semantic Scholar API** (free, no key required) to build a citation checker.

In [ ]:
# Semantic Scholar API - free, no key needed
SEMANTIC_SCHOLAR_BASE = "https://api.semanticscholar.org/graph/v1"

def search_semantic_scholar(query: str, limit: int = 5) -> list:
    """Search Semantic Scholar for papers. Returns list of paper metadata."""
    url = f"{SEMANTIC_SCHOLAR_BASE}/paper/search"
    params = {"query": query, "limit": limit, "fields": "title,year,authors,paperId,externalIds"}
    resp = requests.get(url, params=params)
    if resp.status_code != 200:
        return []
    data = resp.json()
    return data.get("data", [])

def verify_citation(title_or_author: str) -> dict:
    """Check if a paper exists in Semantic Scholar. Returns verification result."""
    results = search_semantic_scholar(title_or_author, limit=3)
    if not results:
        return {"verified": False, "message": "No matching papers found", "candidates": []}
    best = results[0]
    return {
        "verified": True,
        "paperId": best.get("paperId"),
        "title": best.get("title"),
        "year": best.get("year"),
        "authors": [a.get("name") for a in best.get("authors", [])],
        "candidates": results
    }

# Demo: search for a real paper
print("Searching for 'Attention Is All You Need'...")
result = verify_citation("Attention Is All You Need")
print(json.dumps(result, indent=2))

In [ ]:
# The hallucinated citation problem: AI often invents plausible-sounding papers
# Example: a citation that sounds real but may not exist
fake_citations = [
    "Smith et al. (2022) - Deep Learning for Citation Verification",
    "Johnson & Lee (2023) - Multimodal Reasoning in Large Language Models",
    "Attention Is All You Need",  # This one is REAL
]

print("Verifying AI-generated style citations:\n")
for cite in fake_citations:
    r = verify_citation(cite)
    status = "✅ VERIFIED" if r["verified"] else "❌ NOT FOUND"
    print(f"{cite}")
    print(f"  → {status}")
    if r.get("title"):
        print(f"  → Matched: {r['title']} ({r.get('year', '?')})")
    print()

In [ ]:
# Build a function that takes AI output and flags unverifiable claims
import re

def extract_citations(text: str) -> list:
    """Extract citation-like patterns: 'Author et al. (Year)' or 'Author (Year)' or paper titles in quotes."""
    patterns = [
        r'([A-Z][a-z]+(?:\s+et\s+al\.?)?)\s*\((\d{4})\)',  # Author et al. (Year)
        r'"([^"]+)"\s*\([A-Za-z\s]+,\s*\d{4}\)',           # "Title" (Author, Year)
        r'(?:^|\n)\s*[-•]\s*([^(\n]+)\s*\((\d{4})\)',      # - Title (Year)
    ]
    citations = []
    for p in patterns:
        for m in re.finditer(p, text):
            citations.append(m.group(0).strip())
    return list(set(citations))

def flag_unverifiable_claims(text: str) -> dict:
    """Given AI-generated research text, identify claims with unverifiable citations."""
    citations = extract_citations(text)
    results = {"verified": [], "unverifiable": [], "raw_citations": citations}
    
    for cite in citations:
        # Extract searchable part: prefer title when in "Author (Year) - Title" format
        if " - " in cite:
            search_term = cite.split(" - ", 1)[-1].strip()
        else:
            search_term = cite.split("(")[0].strip() if "(" in cite else cite
        if len(search_term) < 5:
            continue
        r = verify_citation(search_term)
        if r["verified"]:
            results["verified"].append({"citation": cite, "match": r.get("title", "")})
        else:
            results["unverifiable"].append(cite)
    
    return results

# Demo with sample AI output
sample_ai_output = """
Key findings on RAG:
- Lewis et al. (2020) - Retrieval-Augmented Generation for Knowledge-Intensive NLP
- "Deep Learning for Citation Verification" (Smith, 2022) suggests automated checks improve accuracy.
"""
flags = flag_unverifiable_claims(sample_ai_output)
print("Extracted citations:", flags["raw_citations"])
print("\nVerified:", flags["verified"])
print("Unverifiable (need manual check):", flags["unverifiable"])

---
## 4. Cross-Validation Across Tools

Different AI research tools can produce **conflicting answers** for the same query. Comparing outputs across tools reveals areas that need primary source verification. Below we build a simple comparison matrix: *claim vs. tool agreement*.

In [ ]:
# Framework: compare outputs from multiple AI tools on the same query
def build_comparison_matrix(query: str, tool_outputs: dict) -> pd.DataFrame:
    """
    tool_outputs: {"Tool A": "claim1. claim2.", "Tool B": "claim1. claim2.", ...}
    Returns a matrix of claims vs. which tools support them.
    """
    # Simplified: extract sentence-level "claims" (split by period)
    all_claims = set()
    for tool, output in tool_outputs.items():
        claims = [c.strip() for c in output.split(".") if len(c.strip()) > 20]
        all_claims.update(claims)
    
    # Build matrix: claim -> {Tool: True/False}
    rows = []
    for claim in list(all_claims)[:10]:  # Limit for display
        row = {"claim": claim[:80] + "..." if len(claim) > 80 else claim}
        for tool, output in tool_outputs.items():
            row[tool] = "✓" if claim[:30] in output or any(c in output for c in claim.split()[:5]) else "—"
        rows.append(row)
    
    return pd.DataFrame(rows)

# Simulated outputs from different tools (in practice, you'd call each API)
simulated_outputs = {
    "ChatGPT": "RAG improves factuality. Retrieval-augmented generation reduces hallucinations. Dense retrieval outperforms BM25 in some settings.",
    "Perplexity": "RAG reduces hallucinations. Hybrid search combines dense and sparse retrieval. Factuality gains depend on retrieval quality.",
    "Claude": "RAG improves factuality when retrieval is accurate. Hallucinations can increase if retrieved docs are irrelevant. BM25 remains competitive."
}

df = build_comparison_matrix("Does RAG reduce hallucinations?", simulated_outputs)
print("Claim vs. Tool Agreement Matrix:\n")
print(df.to_string())

In [ ]:
# Conflicting outputs reveal areas needing primary source verification
# Claims with "—" across tools or mixed ✓/— should be verified manually
print("Interpretation:")
print("- Claims with ✓ across all tools: higher confidence (but still verify key citations)")
print("- Claims with mixed agreement: run primary source check (Semantic Scholar, Google Scholar)")
print("- Claims unique to one tool: treat as hypothesis, not fact")

---
## 5. Research Synthesis Best Practices

### Structuring an AI-Assisted Literature Review

1. **AI generates leads** — Use AI to identify key papers, concepts, and potential gaps.
2. **Human verifies** — Check every citation in Semantic Scholar or Google Scholar.
3. **Human synthesizes** — Write the final narrative; AI assists, does not replace, your analysis.

### The "Trust but Verify" Workflow

| Step | AI Role | Human Role |
|------|---------|------------|
| Discovery | Suggest papers, summarize abstracts | Select relevant papers, filter noise |
| Synthesis | Draft connections, outline structure | Verify claims, add critical analysis |
| Citation | Propose citations | **Verify every one** before including |
| Writing | Suggest phrasing, fill gaps | Final authorship, ensure originality |

In [ ]:
# Template for a verified research brief
def create_research_brief_template(topic: str, verified_sources: list, ai_draft: str) -> str:
    """
    Structure for a research brief that documents AI usage and verification.
    verified_sources: list of dicts with keys: title, authors, year, paperId (from Semantic Scholar)
    """
    sources_str = "\n".join([f"- {s.get('title', '')} ({s.get('year', ''))}" for s in verified_sources])
    return f"""
# Verified Research Brief: {topic}

## AI-Assisted Draft (pre-verification)
{ai_draft[:500]}...

## Verified Sources (checked via Semantic Scholar)
{sources_str}

## Disclosure
This brief was prepared with AI assistance. All citations were verified against Semantic Scholar.
AI tools used: [list tools]. Human verification date: [fill in].
"""

# Example usage
template = create_research_brief_template(
    topic="RAG for factuality",
    verified_sources=[{"title": "Retrieval-Augmented Generation for Knowledge-Intensive NLP", "year": 2020}],
    ai_draft="RAG reduces hallucinations by grounding generation in retrieved documents..."
)
print(template)

---
## 6. Academic Integrity and Disclosure

### What Counts as AI-Assisted vs. AI-Generated Research

| Level | Description | Disclosure |
|-------|-------------|------------|
| **AI-Assisted** | You use AI for brainstorming, literature discovery, or drafting; you verify, edit, and take responsibility | Acknowledge AI tools used; describe verification steps |
| **AI-Generated** | AI produces substantial text you submit with minimal human input | Many institutions require explicit disclosure; may be restricted |
| **AI-Prohibited** | Some assignments (exams, certain papers) forbid AI entirely | Follow course/institution policy |

### Disclosure Templates

**For coursework:**
> "I used [ChatGPT/Claude/Elicit] to identify relevant papers and draft an initial outline. All citations were verified using Semantic Scholar. I wrote the final analysis and take full responsibility for the content."

**For publications:**
> "AI tools were used for literature discovery and drafting. All claims and citations were verified by the authors. No AI-generated text was included without substantial human editing."

**For grants:**
> "AI-assisted literature review: tools used for [specific tasks]. Human verification performed for all cited sources."

### Institutional Policy Considerations

- Check your institution's AI policy (many have adopted guidelines in 2023–2024).
- When in doubt, **disclose more rather than less**.
- Document your verification workflow — it demonstrates rigor.

---
## Exercises

Complete these exercises to reinforce the verification workflow and responsible AI practices.

### Exercise 1: Semantic Scholar vs. LLM

Use the Semantic Scholar API to search for papers on a topic of your choice (e.g., "contrastive learning", "instruction tuning", "multimodal transformers"). Then, ask an LLM (ChatGPT, Claude, or via API) for the same query. Compare:
- Do the papers overlap?
- Did the LLM cite any papers not in the Semantic Scholar results?
- Which source gave you more actionable leads?

In [ ]:
# Exercise 1: Your code here
# 1. Pick a topic and search Semantic Scholar
topic = "contrastive learning"  # Change to your topic
results = search_semantic_scholar(topic, limit=5)
print(f"Semantic Scholar results for '{topic}':")
for i, p in enumerate(results, 1):
    print(f"  {i}. {p.get('title')} ({p.get('year')})")
# 2. Compare with LLM output (use OpenAI cell above or another tool)
# 3. Document overlap and any unverifiable LLM citations

### Exercise 2: Verification Pipeline

Build a verification pipeline that takes a paragraph of AI-generated research text and identifies which claims have verifiable sources. Extend the `flag_unverifiable_claims` function to:
- Handle more citation formats (e.g., numbered references, footnotes)
- Output a summary: "X of Y citations verified"
- Suggest search queries for unverifiable items

In [ ]:
# Exercise 2: Extend the verification pipeline
# Add: more citation formats, summary stats, suggested search queries
def extended_flag_unverifiable(text: str) -> dict:
    result = flag_unverifiable_claims(text)
    total = len(result["verified"]) + len(result["unverifiable"])
    verified = len(result["verified"])
    result["summary"] = f"{verified} of {total} citations verified"
    result["search_suggestions"] = [f'"{c}"' for c in result["unverifiable"]]
    return result
# Test with your own AI-generated paragraph

### Exercise 3: Verified Research Brief

Write a **200-word verified research brief** on a topic of your choice using the workflow from this notebook. Include:
1. A clear research question
2. Key findings with citations
3. Full disclosure of AI tool usage
4. Confirmation that all citations were verified via Semantic Scholar (or similar)

In [ ]:
# Exercise 3: Write your 200-word verified research brief below
# Use create_research_brief_template() and verify all citations with verify_citation()
my_topic = "Your topic here"
my_verified_sources = []  # Populate from Semantic Scholar verification
my_draft = "Your 200-word brief..."
# create_research_brief_template(my_topic, my_verified_sources, my_draft)

---
## Responsible AI Reflection

AI research tools can synthesize hundreds of papers in seconds, but they can also generate convincing citations to papers that don't exist.

**Consider:**
- Does using AI for research make you a **better researcher** or a **lazier** one?
- Where is the line between **"AI-assisted research"** and **"AI-generated research"**?
- What **verification practices** will you commit to for the rest of this semester?

*Reflect in a few sentences. There are no wrong answers — the goal is to develop a personal ethic for AI use in academic work.*

---
## Summary & Next Steps

### Key Takeaways

- **AI research tools** (ChatGPT Deep Research, Claude, Perplexity, Elicit, Semantic Scholar) each have strengths — choose based on task.
- **Prompt design** matters: use structured templates with scope, depth, and format instructions.
- **Verification is mandatory**: AI hallucinates citations. Use Semantic Scholar API (free) to verify every citation.
- **Cross-validate** across tools when answers conflict; primary source verification resolves disputes.
- **Disclose** AI usage appropriately for coursework, publications, and grants.

### Next Steps

- **Week 3**: Responsible AI Foundations — ethics, bias, and safety.
- **Apply this workflow** to every research task this semester. Build the habit of "trust but verify."